In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_from_disk, concatenate_datasets
from sklearn.metrics import average_precision_score, roc_auc_score

print(torch.cuda.device_count(), torch.cuda.get_device_name(0))

/home/shivraj-pg/miniconda3/envs/assignment/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1 NVIDIA L40


In [2]:
MODEL_NAME = "Qwen/Qwen3-8B"
DATA_PATH  = "data/wikibio/sentence_level_split"
RESULT_DIR = "results"
os.makedirs(RESULT_DIR, exist_ok=True)

WIKI_PREFIX = "Here is a short Wikipedia-style biography.\n\n"

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()
print("Model loaded successfully!")

W1104 19:20:53.553000 861158 site-packages/torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1104 19:20:53.579000 861158 site-packages/torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
Loading checkpoint shards: 100%|██████████| 5/5 [00:02<00:00,  1.80it/s]


Model loaded successfully!


### Scoring function

In [5]:
def score_batch(contexts, targets, model, tokenizer):
    ctx_ids = tokenizer(contexts, add_special_tokens=False)["input_ids"]
    tgt_ids = tokenizer(targets,  add_special_tokens=False)["input_ids"]

    B = len(contexts)
    max_len = max(len(c) + len(t) for c, t in zip(ctx_ids, tgt_ids))

    input_ids      = torch.full((B, max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_len), dtype=torch.long)
    target_mask    = torch.zeros((B, max_len), dtype=torch.bool)   
    for i, (c, t) in enumerate(zip(ctx_ids, tgt_ids)):
        L = len(c) + len(t)
        input_ids[i, :L] = torch.tensor(c + t)
        attention_mask[i, :L] = 1
        target_mask[i, len(c):L] = True

    device = model.get_input_embeddings().weight.device
    input_ids, attention_mask, target_mask = input_ids.to(device), attention_mask.to(device), target_mask.to(device)

    with torch.inference_mode():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits[:, :-1, :].float()
        labels = input_ids[:, 1:]           
        tmask  = target_mask[:, 1:]             

        log_probs = F.log_softmax(logits, dim=-1)
        token_logp = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)
        token_entropy = -(log_probs.exp() * log_probs).sum(-1)

        results = []
        for i in range(B):
            m = tmask[i]
            if m.sum() == 0:
                results.append({"mean_logprob": np.nan, "mean_entropy": np.nan, "num_tokens": 0})
            else:
                results.append({
                    "mean_logprob": token_logp[i][m].mean().item(),
                    "mean_entropy": token_entropy[i][m].mean().item(),
                    "num_tokens": int(m.sum().item()),
                })
    return results

## Load data and build (context, target) pairs

In [6]:
dataset = load_from_disk(DATA_PATH)
full = concatenate_datasets([dataset["train"], dataset["test"]]).to_pandas()
full = full.sort_values(["bio_id", "sentence_id"]).reset_index(drop=True) 
print("Total sentences:", len(full), "| positive rate:", round(full["label"].mean(), 4))

Total sentences: 1908 | positive rate: 0.7296


In [7]:
contexts, targets = [], []
for bio_id, g in full.groupby("bio_id", sort=False):
    prev = []              
    for s in g["sentence"]:
        contexts.append(WIKI_PREFIX + " ".join(prev))
        targets.append((" " if prev else "") + s) 
        prev.append(s)

assert len(contexts) == len(full)
print(repr(contexts[1][-120:]), "->", repr(targets[1][:60]))

'Here is a short Wikipedia-style biography.\n\nJohn Russell Reynolds (1820–1876) was an English lawyer, judge, and author.' -> ' He was born in London, the son of a barrister, and was educ'


## Consistency check: batched vs one-at-a-time (small bf16 differences are expected)

In [8]:
idx = list(range(1, 5))  
single  = [score_batch([contexts[i]], [targets[i]], model, tokenizer)[0] for i in idx]
batched = score_batch([contexts[i] for i in idx], [targets[i] for i in idx], model, tokenizer)

for i, a, b in zip(idx, single, batched):
    print(i, f"logprob {a['mean_logprob']:.4f} vs {b['mean_logprob']:.4f} | "
             f"entropy {a['mean_entropy']:.4f} vs {b['mean_entropy']:.4f} | tokens {a['num_tokens']}/{b['num_tokens']}")

1 logprob -0.6500 vs -0.6495 | entropy 0.9485 vs 0.9507 | tokens 26/26
2 logprob -0.5506 vs -0.5585 | entropy 0.6582 vs 0.6602 | tokens 26/26
3 logprob -0.6368 vs -0.6351 | entropy 0.6614 vs 0.6582 | tokens 30/30
4 logprob -1.1245 vs -1.1366 | entropy 0.9450 vs 0.9567 | tokens 14/14


## Score all sentences

In [9]:
BATCH_SIZE = 4
all_scores = []
for start in tqdm(range(0, len(full), BATCH_SIZE)):
    all_scores.extend(score_batch(contexts[start:start + BATCH_SIZE],
                                  targets[start:start + BATCH_SIZE], model, tokenizer))

100%|██████████| 477/477 [01:08<00:00,  6.93it/s]


In [10]:
results_df = pd.concat([full, pd.DataFrame(all_scores)], axis=1)

results_df["score_logprob"] = -results_df["mean_logprob"]
results_df["score_entropy"] = results_df["mean_entropy"]

In [11]:
results_df.to_csv(f"{RESULT_DIR}/task1_wikibio.csv", index=False)

In [12]:
print("Missing:", results_df[["mean_logprob", "mean_entropy"]].isna().sum().to_dict())

Missing: {'mean_logprob': 0, 'mean_entropy': 0}


In [13]:
results_df[["mean_logprob", "mean_entropy"]].describe()

,mean_logprob,mean_entropy
count,1908.000000,1908.000000
mean,-1.053520,0.923898
std,0.519949,0.363694
min,-4.837143,0.039626
25%,-1.331700,0.654128
50%,-0.972862,0.881868
75%,-0.694459,1.150632
max,-0.006545,3.421099


## Final reporting: AUC-PR (full dataset)

In [14]:
y = results_df["label"]
print(f"Random baseline (positive rate): {y.mean():.4f}")
print(f"AUC-PR  mean log-prob : {average_precision_score(y, results_df['score_logprob']):.4f}")
print(f"AUC-PR  mean entropy  : {average_precision_score(y, results_df['score_entropy']):.4f}")

Random baseline (positive rate): 0.7296
AUC-PR  mean log-prob : 0.6747
AUC-PR  mean entropy  : 0.7194


## Additional analysis

In [ ]:
results_df.groupby("label")[["mean_logprob", "mean_entropy", "num_tokens"]].agg(["mean", "median", "std"]).round(3)

mean_logprob               mean_entropy               num_tokens         \
              mean median    std         mean median    std       mean median   
label                                                                           
0           -1.168 -1.069  0.618        0.936  0.891  0.402     26.145   24.0   
1           -1.011 -0.944  0.472        0.919  0.880  0.349     25.988   25.0   

               
          std  
label          
0      12.225  
1      10.071

In [16]:
print("ROC-AUC log-prob:", round(roc_auc_score(y, results_df["score_logprob"]), 4))
print("ROC-AUC entropy :", round(roc_auc_score(y, results_df["score_entropy"]), 4))

ROC-AUC log-prob: 0.4332
ROC-AUC entropy : 0.4994
